# Análisis y Visualización Avanzada

## Contexto empresarial

AndesRetail S.A.C. es una empresa ficticia de retail omnicanal que opera 25 tiendas en cinco ciudades del Perú: Lima, Arequipa, Trujillo, Cusco y Piura.

Durante los últimos dos años, la gerencia ha incrementado la inversión en marketing y ha impulsado el canal digital. Las ventas han crecido, pero la dirección comercial no tiene claridad sobre qué indicadores están realmente relacionados con el crecimiento.

Algunos gerentes consideran que la inversión en marketing explica el incremento de ventas. Otros sostienen que son la cantidad de clientes, los pedidos, la satisfacción y los tiempos de entrega.

Además, el área de analítica ha detectado que varios indicadores parecen comportarse de forma muy similar, por lo que podría existir multicolinealidad o redundancia entre algunos KPI.

La empresa solicita al equipo de analistas realizar un estudio exploratorio y presentar los principales hallazgos a la Gerencia General.

## Objetivo

Su responsabilidad será analizar los indicadores comerciales de la empresa utilizando Python, Pandas, Matplotlib y Seaborn.

**¿Qué variables están más relacionadas con las ventas y qué indicadores podrían estar proporcionando información redundante?**

## Data

Archivo: `andes_retail_kpis.csv`

| Campo                 | Descripción                       |
| --------------------- | --------------------------------- |
| `periodo`             | Mes analizado                     |
| `tienda_id`           | Código de la tienda               |
| `ciudad`              | Ciudad                            |
| `canal_principal`     | Tienda, Mixto u Online            |
| `ventas_mensuales`    | Ventas mensuales en soles         |
| `clientes`            | Clientes atendidos                |
| `pedidos`             | Número de pedidos                 |
| `visitas_web`         | Visitas al sitio web              |
| `inversion_marketing` | Inversión mensual en marketing    |
| `descuento_promedio`  | Porcentaje promedio de descuento  |
| `ticket_promedio`     | Importe promedio de compra        |
| `satisfaccion`        | Satisfacción del cliente de 1 a 5 |
| `reclamos`            | Reclamos recibidos                |
| `tiempo_entrega_dias` | Tiempo promedio de entrega        |
| `costo_logistico`     | Costo logístico mensual           |
| `tasa_conversion`     | Porcentaje de conversión          |


In [ ]:
## 1. Cargando librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy  

In [ ]:
## 2. Cargando el dataset
df = pd.read_csv("./data/andes_retail_kpis.csv")
df.head()


In [ ]:
## Tamaño del dataset
df.shape

In [ ]:
## Información del dataset
df.info()

In [ ]:
## Estadísticas descriptivas
df.describe()

In [ ]:
## Columnas categoricas
categorical_columns = df.select_dtypes(include=['object']).columns
categorical_columns

In [ ]:
## Convertir la columna periodo a datatime
df['periodo'] = pd.to_datetime(df['periodo'], format='%Y-%m-%d')

In [ ]:
## Generando matriz de Pearson
columnas_numericas = df.select_dtypes(include=['float64', 'int64']).columns
correlation_pearson = df[columnas_numericas].corr(method='pearson')
correlation_pearson

In [ ]:
## Funcion para clasificar la correlación (alta > 0.7, moderada 0.3-0.7, baja < 0.3)
def clasificar_correlacion(valor):
    if abs(valor) > 0.7:
        return 'alta'
    elif abs(valor) > 0.3:
        return 'moderada'
    else:
        return 'baja'

In [ ]:
## Recorriendo la matriz de correlación y clasificando los valores
matriz_pearson = correlation_pearson.apply(lambda columna: columna.map(clasificar_correlacion))
matriz_pearson

In [ ]:
## Generando matriz de spearman
correlation_spearman = df[columnas_numericas].corr(method='spearman')
correlation_spearman

In [ ]:
## Recorriendo la matriz de correlación y clasificando los valores
matriz_spearman = correlation_spearman.apply(lambda columna: columna.map(clasificar_correlacion))
matriz_spearman

In [ ]:
## Generando matriz de kendall
correlation_kendall = df[columnas_numericas].corr(method='kendall')
correlation_kendall

In [ ]:
## Recorriendo la matriz de correlación y clasificando los valores
matriz_kendall = correlation_kendall.apply(lambda columna: columna.map(clasificar_correlacion))
matriz_kendall

In [ ]:
## Tabla de correlaciones por pares de variables
pares = {
    'Clientes / Pedidos': ('clientes', 'pedidos'),
    'Marketing / Visitas': ('inversion_marketing', 'visitas_web'),
    'Satisfacción / Reclamos': ('satisfaccion', 'reclamos'),
}

def interpretar_correlacion(v1, v2, valor):
    if abs(valor) > 0.7:
        fuerza = 'fuerte'
    elif abs(valor) > 0.3:
        fuerza = 'moderada'
    else:
        fuerza = 'débil'

    if valor > 0:
        return f'Relación {fuerza} y positiva: a mayor {v1}, mayor {v2}.'
    else:
        return f'Relación {fuerza} y negativa: a mayor {v1}, menor {v2}.'

tabla_correlaciones = pd.DataFrame({
    'variables': list(pares.keys()),
    'Pearson': [matriz_pearson.loc[v1, v2] for v1, v2 in pares.values()],
    'Spearman': [matriz_spearman.loc[v1, v2] for v1, v2 in pares.values()],
    'Kendall': [matriz_kendall.loc[v1, v2] for v1, v2 in pares.values()],
})

tabla_correlaciones['interpretacion'] = [
    interpretar_correlacion(v1, v2, correlation_pearson.loc[v1, v2])
    for v1, v2 in pares.values()
]

tabla_correlaciones